# Online Payments Fraud Detection

The goal is to classify online payment transactions as fraud or not fraud. The dataset is the PaySim simulation of mobile money transactions ([Kaggle](https://www.kaggle.com/datasets/ealaxi/paysim1), saved locally as `dataset.csv`, 6.36 million rows, not included in the repository because of its size).

The reference solution trains a Decision Tree on four features and reports only accuracy (0.9997). Fraud is about 0.13% of the data, so accuracy says almost nothing here. This notebook evaluates the models with a confusion matrix, precision, recall and F1, and handles the class imbalance explicitly.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
os.chdir('/mnt/c/Users/myama/OneDrive/Belgeler/ai-1/Assignments/14-Specialize in Data Science/Classification/online_payments_fraud_detection')

In [2]:
#pip install imbalanced-learn


## Data Loading

In [3]:
df = pd.read_csv("dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 706.2 MB


## Exploratory Data Analysis

In [4]:
print(df["isFraud"].value_counts())
print("fraud share (%):", df["isFraud"].mean() * 100)
print("accuracy of always predicting 'not fraud':", 1 - df["isFraud"].mean())

isFraud
0    6354407
1       8213
Name: count, dtype: int64
fraud share (%): 0.12908204481801522
accuracy of always predicting 'not fraud': 0.9987091795518198


In [5]:
df.groupby("type")["isFraud"].agg(["sum", "count"])

,sum,count
type,,
CASH_IN,0,1399284
CASH_OUT,4116,2237500
DEBIT,0,41432
PAYMENT,0,2151495
TRANSFER,4097,532909


Fraud is only 0.13% of the transactions, so a model that always answers "not fraud" already reaches an accuracy of about 0.9987, and never detects a single fraud (recall 0). The reference accuracy of 0.9997 has to be read against this baseline. Fraud also occurs only in `CASH_OUT` and `TRANSFER` transactions, and never in `CASH_IN`, `DEBIT` or `PAYMENT`.

## Feature Engineering

- `type` is one-hot encoded with `get_dummies`. The reference maps it to the numbers 1 to 5, which makes the model see a false order between the transaction types.
- `nameOrig` and `nameDest` are dropped: they are account identifiers with millions of distinct values and do not generalize to new accounts.
- `isFlaggedFraud` is dropped: it is the output of the simulator's own rule (only 16 rows are flagged), not an independent feature.
- `step` (the hour of the simulation) is dropped. A model that uses it learns when fraud happened, not what fraud looks like, and cannot be applied to a later period. The cross-validation in the Evaluation section shows this.

The data is split 80/20. The split is stratified so that both sets keep the 0.13% fraud share.

In [6]:
x = pd.get_dummies(df.drop(columns=["isFraud", "nameOrig", "nameDest", "isFlaggedFraud", "step"]), columns=["type"], drop_first=True)
y = df["isFraud"]

# stratify keeps the fraud share equal in both sets (parameter found while researching imbalanced data, not covered in class)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print("train:", x_train.shape, "fraud:", y_train.sum(), f"({y_train.mean() * 100:.3f}%)")
print("test: ", x_test.shape, "fraud:", y_test.sum(), f"({y_test.mean() * 100:.3f}%)")

train: (5090096, 9) fraud: 6570 (0.129%)
test:  (1272524, 9) fraud: 1643 (0.129%)


## Handling Class Imbalance

SMOTE creates synthetic minority examples by interpolating between a fraud sample and its nearest fraud neighbours. Applied alone to 5 million training rows it would generate about 5 million synthetic frauds from only 6,570 real ones, and make training very slow. So the majority class is first randomly undersampled to 10 times the number of frauds, then SMOTE brings the frauds up to the same size. Resampling is applied to the training set only. The test set keeps the real 0.13% fraud share, otherwise the evaluation would not reflect real use.

In [7]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

x_under, y_under = RandomUnderSampler(sampling_strategy=0.1, random_state=42).fit_resample(x_train, y_train)
x_res, y_res = SMOTE(random_state=42).fit_resample(x_under, y_under)

print("before:", y_train.value_counts().to_dict())
print("after: ", y_res.value_counts().to_dict())

before: {0: 5083526, 1: 6570}
after:  {0: 65700, 1: 65700}


## Model Training

Four classifiers from the course are trained on the resampled training set and evaluated on the untouched test set. Accuracy is reported for reference, but the comparison is based on precision, recall and F1 of the fraud class.

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

results, predictions = {}, {}
for name, model in models.items():
    pred = model.fit(x_res, y_res).predict(x_test)
    predictions[name] = pred
    report = classification_report(y_test, pred, output_dict=True)["1"]
    results[name] = {"Accuracy": accuracy_score(y_test, pred), "Precision": report["precision"],
                     "Recall": report["recall"], "F1": report["f1-score"]}

pd.DataFrame(results).T.round(4)

,Accuracy,Precision,Recall,F1
Logistic Regression,0.9489,0.0233,0.9416,0.0454
Decision Tree,0.9957,0.2322,0.9957,0.3765
Random Forest,0.9959,0.2403,0.9957,0.3872
Gradient Boosting,0.9838,0.0737,0.9957,0.1372


The resampled models reach a recall above 0.99, but their precision is very low: they were trained on a 50/50 class balance, and on the real test data (0.13% fraud) that produces a large number of false alarms. Accuracy (0.9957 for the Decision Tree) is even below the 0.9987 of always predicting "not fraud". The same models are now trained on the original, unresampled training set. Gradient Boosting is left out here because it is too slow on 5 million rows, and Logistic Regression because it is not converging on unscaled amounts of this size.

In [9]:
raw_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
}

results_raw = {}
for name, model in raw_models.items():
    pred = model.fit(x_train, y_train).predict(x_test)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    report = classification_report(y_test, pred, output_dict=True)["1"]
    results_raw[name] = {"Accuracy": accuracy_score(y_test, pred), "Precision": report["precision"],
                         "Recall": report["recall"], "F1": report["f1-score"], "TP": tp, "FN": fn, "FP": fp}

pd.DataFrame(results_raw).T.round(4)

,Accuracy,Precision,Recall,F1,TP,FN,FP
Decision Tree,0.9998,0.9079,0.9002,0.904,1479.0,164.0,150.0
Random Forest,0.9997,0.9642,0.7876,0.867,1294.0,349.0,48.0


## Evaluation

Both approaches side by side on the same test set (1,643 frauds among 1,272,524 transactions). The confusion counts of the resampled models are added so that the number of false alarms is visible.

In [10]:
def confusion_counts(pred):
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {"TP": tp, "FN": fn, "FP": fp}

results_resampled = pd.DataFrame(results).T.join(pd.DataFrame({n: confusion_counts(p) for n, p in predictions.items()}).T)
comparison = pd.concat({"Undersampling + SMOTE": results_resampled, "No resampling": pd.DataFrame(results_raw).T})
comparison.round(4)

Accuracy  Precision  Recall  \
Undersampling + SMOTE Logistic Regression    0.9489     0.0233  0.9416   
                      Decision Tree          0.9957     0.2322  0.9957   
                      Random Forest          0.9959     0.2403  0.9957   
                      Gradient Boosting      0.9838     0.0737  0.9957   
No resampling         Decision Tree          0.9998     0.9079  0.9002   
                      Random Forest          0.9997     0.9642  0.7876   

                                               F1      TP     FN       FP  
Undersampling + SMOTE Logistic Regression  0.0454  1547.0   96.0  64906.0  
                      Decision Tree        0.3765  1636.0    7.0   5411.0  
                      Random Forest        0.3872  1636.0    7.0   5172.0  
                      Gradient Boosting    0.1372  1636.0    7.0  20571.0  
No resampling         Decision Tree        0.9040  1479.0  164.0    150.0  
                      Random Forest        0.8670  1294.0  349.0     48.0

Resampling raised the recall (0.90 to 0.996 for the Decision Tree) but made the precision collapse (0.91 to 0.23), so the F1 fell from 0.90 to 0.38. Without resampling, the Decision Tree flags 150 legitimate transactions and misses 164 frauds. With resampling, it flags about 5,400 legitimate transactions to miss only 7 frauds. In this dataset the fraud pattern is clear enough for the model to learn it from the original imbalance, so the synthetic examples do not help.

The reference model is reproduced below with the same evaluation, to compare on equal terms: four features, `type` mapped to the numbers 1 to 5, a 90/10 split without stratification, and a Decision Tree.

In [11]:
ref_type = df["type"].map({"CASH_OUT": 1, "PAYMENT": 2, "CASH_IN": 3, "TRANSFER": 4, "DEBIT": 5})
ref_x = pd.concat([ref_type, df[["amount", "oldbalanceOrg", "newbalanceOrig"]]], axis=1)
ref_x_train, ref_x_test, ref_y_train, ref_y_test = train_test_split(ref_x, df["isFraud"], test_size=0.10, random_state=42)

ref_pred = DecisionTreeClassifier().fit(ref_x_train, ref_y_train).predict(ref_x_test)
print("accuracy:", accuracy_score(ref_y_test, ref_pred))
print(classification_report(ref_y_test, ref_pred, digits=4))

accuracy: 0.9997280994307377
              precision    recall  f1-score   support

           0     0.9998    0.9999    0.9999    635445
           1     0.9035    0.8825    0.8929       817

    accuracy                         0.9997    636262
   macro avg     0.9517    0.9412    0.9464    636262
weighted avg     0.9997    0.9997    0.9997    636262



In [12]:
tree = raw_models["Decision Tree"]
pd.Series(tree.feature_importances_, index=x.columns).sort_values(ascending=False).round(3)

oldbalanceOrg     0.440
newbalanceDest    0.228
amount            0.169
oldbalanceDest    0.085
newbalanceOrig    0.060
type_TRANSFER     0.017
type_CASH_OUT     0.000
type_DEBIT        0.000
type_PAYMENT      0.000
dtype: float64

The reproduced reference model reaches an F1 of 0.89 on its own 90/10 split, close to the 0.90 of our Decision Tree, but the two test sets are different (817 versus 1,643 frauds), so a single split cannot tell whether the gap is real. A 5-fold cross-validation compares feature sets on identical folds. `cross_validate` was found while researching a fair comparison and is not covered in class. It returns the scores of every fold in one call, which also avoids writing a loop over splits.

The data is sorted by `step`, so the default folds are consecutive time blocks: each fold is tested on a period the model has not seen. This is also how the model would be used in practice (trained on the past, applied to the future). The fourth set adds `step` back to show why it was dropped.

In [13]:
from sklearn.model_selection import cross_validate

four_cols = ["amount", "oldbalanceOrg", "newbalanceOrig"] + [c for c in x.columns if c.startswith("type_")]
feature_sets = {"Reference (4 features, type 1-5)": ref_x, "4 features, type one-hot": x[four_cols],
                "9 features (ours)": x, "10 features (ours + step)": x.join(df["step"])}

cv_results = {name: cross_validate(DecisionTreeClassifier(random_state=42), features, y, cv=5, n_jobs=-1,
                                   scoring=["precision", "recall", "f1"]) for name, features in feature_sets.items()}

pd.DataFrame({name: {metric: f"{r[f'test_{metric}'].mean():.3f} +/- {r[f'test_{metric}'].std():.3f}"
                     for metric in ["precision", "recall", "f1"]} for name, r in cv_results.items()}).T

,precision,recall,f1
"Reference (4 features, type 1-5)",0.888 +/- 0.016,0.876 +/- 0.007,0.882 +/- 0.011
"4 features, type one-hot",0.886 +/- 0.016,0.877 +/- 0.009,0.881 +/- 0.012
9 features (ours),0.905 +/- 0.014,0.891 +/- 0.012,0.898 +/- 0.012
10 features (ours + step),0.545 +/- 0.360,0.718 +/- 0.223,0.515 +/- 0.323


## Conclusion

**Reference approach.** The reference trains a Decision Tree on four features (`type` mapped to 1 to 5, `amount`, `oldbalanceOrg`, `newbalanceOrig`) and reports one number, an accuracy of 0.9997. Fraud is 0.13% of the 6.36 million transactions, so always predicting "not fraud" already gives 0.9987 and detects no fraud at all. Reproducing the reference confirmed its accuracy (0.99973) and showed that the model is in fact reasonable (fraud precision 0.90, recall 0.89), but this cannot be seen from accuracy alone.

**Data.** Fraud occurs only in `CASH_OUT` and `TRANSFER` transactions. `nameOrig`, `nameDest` and `isFlaggedFraud` were dropped, and `type` was one-hot encoded instead of mapped to 1 to 5.

**Class imbalance.** Random undersampling followed by SMOTE, applied to the training set only, raised the recall to 0.996 but dropped the precision to 0.23 to 0.24 (Decision Tree and Random Forest), which means about 5,400 false alarms to miss 7 frauds. Trained on the original data, the Decision Tree reaches a precision of 0.91 and a recall of 0.90 (150 false alarms, 164 missed frauds). The fraud pattern (accounts emptied through `TRANSFER` and `CASH_OUT`, visible in `oldbalanceOrg`, `newbalanceDest` and `amount`, which carry most of the feature importance) is clear enough that oversampling is not needed. SMOTE was mentioned in the course but not demonstrated, and it hurt the precision here.

| Model | Precision | Recall | F1 | Missed frauds | False alarms |
|---|---|---|---|---|---|
| Decision Tree, no resampling | 0.908 | 0.900 | 0.904 | 164 | 150 |
| Random Forest, no resampling | 0.964 | 0.788 | 0.867 | 349 | 48 |
| Decision Tree, undersampling + SMOTE | 0.232 | 0.996 | 0.377 | 7 | 5,411 |
| Random Forest, undersampling + SMOTE | 0.240 | 0.996 | 0.387 | 7 | 5,172 |
| Gradient Boosting, undersampling + SMOTE | 0.074 | 0.996 | 0.137 | 7 | 20,571 |
| Logistic Regression, undersampling + SMOTE | 0.031 | 0.879 | 0.060 | 199 | 45,358 |

**Time feature.** `step` looks harmless with a random split, but the data is ordered in time and fraud is unevenly spread over it. With `step`, a 5-fold cross-validation on consecutive time blocks gives an F1 of 0.52 with a standard deviation of 0.32 (0.003 on the last block). Without it, the F1 is 0.90 with a standard deviation of 0.01. The model with `step` memorizes when fraud happened and cannot be applied to a later period, so `step` was dropped.

**Feature sets.** In the same cross-validation, the F1 was 0.882 for the reference features, 0.881 for the same four features with one-hot `type`, and 0.898 for our nine features. The extra destination balance features give a small gain. The gap to the reference is about one standard deviation of the fold scores, so it should not be overstated.

**Limitations.** The dataset is a simulation (PaySim), and its fraud pattern is simpler than in real payment data. The Decision Tree still misses about 10% of the frauds, which would need a threshold analysis or a cost-based comparison of false alarms and missed frauds. Gradient Boosting and Logistic Regression were only trained on the resampled data because of their cost and convergence on 5 million unscaled rows.
